# Deligne–Mumford compactification spike — research highlights

This notebook is a **mathematician-facing tour** of the combinatorial model of
$\overline{\mathcal M}_{g,n}$ implemented in `dm_moduli_spike`. It ignores
enumeration backends and implementation plumbing; the goal is to show how the
owned vocabulary lets you *write like the geometry*.

**Semantic layers (never conflated):**

| object | meaning |
| --- | --- |
| `StableGraphType` | stable dual graph **up to isomorphism** — indexes a stratum |
| `StableGraph` | **labeled** half-edge representative — where contractions live |
| `DMStratum` | the stratum $\mathcal M_\Gamma \subset \mathcal M_{g,n}$ |
| `QuotientStackSignature` | $[\prod_v \mathcal M_{w(v),n_v} / \operatorname{Aut}(\Gamma)]$ |
| `ClutchingDatum` | clutching along $\prod_v \overline{\mathcal M}_{w(v),n_v} \to \overline{\mathcal M}_{g,n}$ |

**Prerequisite:** repository `.envrc` puts `computations/experiments` on
`PYTHONPATH`, so `import dm_moduli_spike` works from Sage without path hacks.

In [ ]:
from dm_moduli_spike import *

def strata_by_codim(stratification):
    """List boundary types rank by rank (codimension = number of nodes)."""
    for codim, bucket in enumerate(stratification.rank_buckets()):
        print(f"codimension {codim}  ({len(bucket)} type{'s' if len(bucket) != 1 else ''})")
        for stratum in bucket:
            gamma = stratum.curve_type()
            print(f"    Γ = {gamma}   (genus profile {gamma.vertex_genera()}, |Aut| = {gamma.automorphism_number()})")
        print()

## 1. Choose an ambient space

Everything is anchored in a single parent
$\overline{\mathcal M}_{g,n}$. We begin with the first interesting boundary
case beyond the elliptic line: **$\overline{\mathcal M}_{1,2}$** (dimension
$1$, one nodal boundary divisor).

In [ ]:
M = DMCompactificationModel(1, 2)
M

Stable dual graph types form a Sage parent `StableGraphTypes(g,n)`. A type is
built from vertex genera, marking blocks, and edges — then **canonicalized**
automatically, so relabeling vertices or markings does not change the
isomorphism class.

In [ ]:
Types = M.graph_types()

# The "theta" graph: two parallel nonseparating nodes on two rational components.
theta = Types.from_vertices(
    genera=(0, 0),
    markings=((1,), (2,)),
    edges=((0, 1), (0, 1)),
)
theta

## 2. The stratification poset

The spike enumerates **all** stable graph types by codimension and recovers
**covers** by one-edge contraction. The result is a finite poset in the
**specialization order**: generic below special, rank = number of nodes.

For $\overline{\mathcal M}_{1,2}$ this poset is Markwig's $Y_{3,2}$ (five
types, two boundary points at codimension $2$).

In [ ]:
S = M.stratification(backend="pure-sage")
print(S)
print("exhaustive:", S.is_exhaustive(), "   full rank support:", S.has_full_rank_support())
print()
strata_by_codim(S)

In [ ]:
P = S.specialization_poset()
print(P)
print("cover relations (generic → special):")
for generic, special in P.cover_relations():
    print(f"  {generic.curve_type()}  ≺  {special.curve_type()}")

Named types in the literature can be recovered **semantically** (by genus
profile and marking placement), without hard-coding internal vertex numbers:

In [ ]:
type_E = S.find_unique_type(
    vertex_genera=(0, 0),
    edge_multiset=((0, 1, 2),),  # two parallel edges
    marking_blocks=((1,), (2,)),
).curve_type()

assert type_E == theta
print("Recovered Markwig type E:", type_E)

## 3. Contractions and edge orbits

On a **labeled** representative $\widetilde\Gamma$, contracting an internal
edge is a morphism $\widetilde\Gamma \to \widetilde{\Gamma'}$.

At the **type** level, elementary contractions come in
$\operatorname{Aut}(\Gamma)$-orbits. Orbit size matters: the two parallel
edges of type E are exchanged by the involution of $\operatorname{Aut}(\Gamma)$,
so there is **one** orbit of size $2$, not two orbits with the same target.

In [ ]:
for orbit in theta.elementary_contraction_orbits():
    print(orbit)
    print("  target type:", orbit.target())
    print("  witness on labeled graph:", orbit.representative())

The poset records covers by **target type**; several distinct orbits may
contribute to the same cover. Specialization comparability is the existential
question "does $\Gamma$ admit a contraction to $\Gamma'$?":

In [ ]:
smooth = M.graph_types().smooth()
print("θ contracts to smooth:", theta.contracts_to(smooth))
print("smooth ≤ θ in specialization order:", P.is_lequal(M.stratum(smooth), M.stratum(theta)))

## 4. Stack presentations and clutching

A stratum is not merely its dual graph. The spike exposes the **quotient stack
signature** and the **clutching datum** in the standard factorization language.

In [ ]:
stratum_theta = M.stratum(theta)
stratum_theta

open_stack = stratum_theta.open_stack_presentation()
print(open_stack)
print("moduli factors:", open_stack.product())
print("|Aut(Γ)|:", open_stack.group_order())

clutching = stratum_theta.clutching_morphism()  # alias: same as ClutchingDatum
print(clutching)

### Local coordinates on each factor

Each vertex $v$ carries a factor $\overline{\mathcal M}_{w(v), n_v}$ with
$n_v$ **local special points**: external legs plus one branch per incident
node. The spike names these uniformly as `FactorSlot`s.

Example: the nodal stratum in $\overline{\mathcal M}_{1,1}$ has one factor
$\overline{\mathcal M}_{0,3}$; the $\mathbb Z/2$ automorphism fixes the
marking slot and swaps the two node branches.

In [ ]:
M11 = DMCompactificationModel(1, 1)
nodal = next(s for s in M11.stratification().strata(codim=1) if s.codimension() == 1)
cd = nodal.clutching_morphism()

print("Clutching:", cd)
print("Factor slots:", cd.factor_slots()[0])

slot_perm = cd.automorphism_action().on_factor_slots()[0][0]
print("Aut action on slot indices:", slot_perm, "  (marking fixed, branches swapped)")

For a separating boundary point in $\overline{\mathcal M}_{0,4}$ (type
$12|34$), each factor has three slots; the gluing map pairs the **third**
slot on each side — the node branches.

In [ ]:
M04 = DMCompactificationModel(0, 4)
split = M04.graph_types().from_vertices(
    genera=(0, 0),
    markings=((1, 2), (3, 4)),
    edges=((0, 1),),
)
cd = M04.stratum(split).clutching_morphism()

external_slots, node_pairings = cd.gluing_map()
print("External markings → slots:")
for label, slot in enumerate(external_slots, start=1):
    print(f"  {label} ↦ factor {slot.vertex}, local index {slot.local_index}")
print("Node pairings:", node_pairings)

## 5. Genus zero: compatible splits

When $g=0$, stable graph types correspond to **stable partitions** of the
marking set. The specialization poset of $\overline{\mathcal M}_{0,n}$ is the
face poset of the split system ( tested in the suite against an independent
oracle). For $n=4$ one sees three maximal boundary points above the smooth
stratum.

In [ ]:
M04 = DMCompactificationModel(0, 4)
P04 = M04.stratification().specialization_poset()
print(P04)
print("maximal boundary strata:")
for s in P04.maximal_elements():
    print(" ", s.curve_type(), "   split system:", s.curve_type().split_system())

---

**Where to go next.** The spike is intentionally combinatorial: it gives
correct stratification posets, contraction orbits, automorphism actions, and
clutching coordinates — not Hodge bundles or Gromov–Witten invariants. For
full enumeration at larger $(g,n)$, install `admcycles` and call
`M.stratification(backend="auto")`; covers are always recovered by
contraction on canonical representatives, independent of the enumerator.